# Chapter 8 — Introduction to Differential Equations

This notebook develops ordinary and partial differential equations through interactive experiments. Its guiding principle is

$$
\boxed{
\text{formal method}
\longrightarrow
\text{candidate}
\longrightarrow
\text{direct verification}
}
$$

Topics include initial-value problems, Picard iteration, first- and second-order ODEs, Euler's method, power series, matrix exponentials, characteristics, heat, wave and Laplace equations, Black–Scholes, and finite-difference stability.

All mathematical Markdown uses dollar delimiters for reliable MathJax rendering.


## 1. Computational setup

For

$$
y'=f(t,y),\qquad y(t_0)=y_0,
$$

the routines below provide Euler and fourth-order Runge–Kutta approximations. Additional helpers support cumulative integration, Fourier-grid heat flow, and the normal distribution function.


In [ ]:
# Core numerical, plotting, and interactive tools.
import math
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, HTML, Math, clear_output

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (9, 4.8), "font.size": 11})

PI = np.pi
TWO_PI = 2.0 * PI


def rk4_scalar(rhs, t0, y0, t1, steps):
    """Compute a scalar IVP reference solution by classical RK4."""
    t = np.linspace(t0, t1, steps + 1)
    y = np.empty(steps + 1, dtype=float)
    y[0] = y0
    h = (t1 - t0) / steps
    for n in range(steps):
        tn, yn = t[n], y[n]
        k1 = rhs(tn, yn)
        k2 = rhs(tn + h / 2, yn + h * k1 / 2)
        k3 = rhs(tn + h / 2, yn + h * k2 / 2)
        k4 = rhs(tn + h, yn + h * k3)
        y[n + 1] = yn + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6
    return t, y


def euler_scalar(rhs, t0, y0, t1, steps):
    """Compute the explicit Euler approximation."""
    t = np.linspace(t0, t1, steps + 1)
    y = np.empty(steps + 1, dtype=float)
    y[0] = y0
    h = (t1 - t0) / steps
    for n in range(steps):
        y[n + 1] = y[n] + h * rhs(t[n], y[n])
    return t, y


def cumulative_trapezoid(x, values):
    """Cumulative trapezoidal integral with zero initial value."""
    result = np.zeros_like(values, dtype=float)
    result[1:] = np.cumsum(0.5 * (values[1:] + values[:-1]) * np.diff(x))
    return result


def normal_cdf(x):
    """Standard normal distribution function without SciPy."""
    x = np.asarray(x, dtype=float)
    return 0.5 * (1 + np.vectorize(math.erf)(x / np.sqrt(2)))


def time_grid(half_width=15.0, samples=2**13):
    return np.linspace(-half_width, half_width, samples, endpoint=False)


def continuous_fft(t, values):
    """Approximate the Fourier transform with angular frequency."""
    dt = t[1] - t[0]
    omega = TWO_PI * np.fft.fftshift(np.fft.fftfreq(t.size, d=dt))
    transform = dt * np.fft.fftshift(np.fft.fft(np.fft.ifftshift(values)))
    return omega, transform


def continuous_ifft(omega, transform):
    """Invert data sampled on the matching FFT frequency grid."""
    domega = omega[1] - omega[0]
    return (transform.size * domega / TWO_PI) * np.fft.fftshift(
        np.fft.ifft(np.fft.ifftshift(transform))
    )


display(HTML(
    "<div style='padding:10px;border-left:4px solid #2a6fbb;background:#eef6ff'>"
    "<b>Setup complete.</b> ODE, PDE, Fourier-grid, and visualization helpers are ready."
    "</div>"
))


## 2. Initial-value problems and slope fields

A solution curve must have slope $f(t,y)$ at every point:

$$
y'(t)=f(t,y(t)).
$$

The equivalent Volterra equation,

$$
y(t)=y_0+\int_{t_0}^{t}f(s,y(s))\,ds,
$$

is the foundation of Picard iteration. Select an equation and compare its direction field with an RK4 trajectory.


In [ ]:
# Plot a slope field and one numerical solution curve.
slope_equation = widgets.Dropdown(
    options=["y - t^2 + 1", "Logistic y(1-y)", "-2ty", "|y|^(2/3)"],
    description="Equation:",
)
slope_y0 = widgets.FloatSlider(value=0.5, min=-2.0, max=2.0, step=0.1, description="y(0):")
slope_horizon = widgets.FloatSlider(value=3.0, min=1.0, max=6.0, step=0.25, description="T:")
slope_button = widgets.Button(description="Draw slope field", button_style="primary", icon="project-diagram")
slope_output = widgets.Output()


def selected_rhs(name):
    if name == "y - t^2 + 1":
        return lambda t, y: y - t * t + 1
    if name == "Logistic y(1-y)":
        return lambda t, y: y * (1 - y)
    if name == "-2ty":
        return lambda t, y: -2 * t * y
    return lambda t, y: np.abs(y) ** (2 / 3)


def draw_slope_field(_=None):
    """Display normalized tangent directions and the selected IVP."""
    with slope_output:
        clear_output(wait=True)
        rhs = selected_rhs(slope_equation.value)
        horizon = slope_horizon.value
        t_mesh = np.linspace(0, horizon, 23)
        y_mesh = np.linspace(-2.5, 3.0, 23)
        T, Y = np.meshgrid(t_mesh, y_mesh)
        slopes = rhs(T, Y)
        norm = np.sqrt(1 + slopes * slopes)
        t_curve, y_curve = rk4_scalar(rhs, 0, slope_y0.value, horizon, 1000)

        fig, ax = plt.subplots()
        ax.quiver(T, Y, 1 / norm, slopes / norm, color="#9e9e9e", angles="xy", pivot="mid")
        ax.plot(t_curve, y_curve, color="#1565c0", linewidth=2.4, label="RK4 trajectory")
        ax.scatter([0], [slope_y0.value], color="#c43c39", zorder=5, label="initial value")
        ax.set_xlim(0, horizon)
        ax.set_ylim(-2.5, 3.0)
        ax.set_xlabel("t")
        ax.set_ylabel("y")
        ax.legend()
        ax.set_title("Direction field and IVP trajectory")
        plt.show()


slope_button.on_click(draw_slope_field)
display(widgets.VBox([
    widgets.HBox([slope_equation, slope_y0, slope_horizon]),
    slope_button,
    slope_output,
]))
draw_slope_field()


## 3. Picard iteration and failure of uniqueness

Picard iteration is

$$
y_{n+1}(t)=y_0+\int_{t_0}^{t}f(s,y_n(s))\,ds.
$$

For $y'=y$, $y(0)=1$, it produces

$$
y_n(t)=\sum_{k=0}^{n}\frac{t^k}{k!}\longrightarrow e^t.
$$

By contrast, $y'=\sqrt{|y|}$, $y(0)=0$, has delayed solutions: they remain zero until $t=c$ and then equal $(t-c)^2/4$. Continuity gives existence, but the Lipschitz condition needed for uniqueness fails at $y=0$.


In [ ]:
# Compare Picard convergence with a non-unique IVP.
picard_mode = widgets.ToggleButtons(options=["Picard iteration", "Non-uniqueness"], description="Experiment:")
picard_order = widgets.IntSlider(value=5, min=0, max=14, description="Iterations:")
picard_delay = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.1, description="Delay c:")
picard_button = widgets.Button(description="Run experiment", button_style="info", icon="redo")
picard_output = widgets.Output()


def run_picard(_=None):
    with picard_output:
        clear_output(wait=True)
        if picard_mode.value == "Picard iteration":
            t = np.linspace(-1.5, 2, 700)
            approximation = sum(t**k / math.factorial(k) for k in range(picard_order.value + 1))
            error = np.max(np.abs(approximation - np.exp(t)))
            fig, ax = plt.subplots()
            ax.plot(t, np.exp(t), color="black", label="exact exp(t)")
            ax.plot(t, approximation, color="#1565c0", label=f"iterate n={picard_order.value}")
            ax.legend()
            plt.show()
            display(HTML(f"<b>Maximum displayed error:</b> {error:.3e}"))
        else:
            t = np.linspace(0, 5, 800)
            fig, ax = plt.subplots()
            for delay in sorted(set([0.0, picard_delay.value, 2.5])):
                solution = np.where(t <= delay, 0.0, (t - delay) ** 2 / 4)
                ax.plot(t, solution, label=f"delay c={delay:.1f}")
            ax.legend()
            ax.set_title("Different solutions with the same initial value")
            plt.show()
            display(HTML("<b>Conclusion:</b> local uniqueness fails because the right-hand side is not locally Lipschitz at y=0."))


picard_button.on_click(run_picard)
display(widgets.VBox([picard_mode, widgets.HBox([picard_order, picard_delay]), picard_button, picard_output]))
run_picard()


## 4. Enter an ODE and obtain a symbolic solution

Enter the right-hand side of

$$
y'(t)=f(t,y(t)),\qquad y(t_0)=y_0.
$$

The cell uses SymPy to find a symbolic solution, displays it with MathJax, and substitutes the candidate back into the original equation. Use syntax such as <code>t-y</code>, <code>y*(1-y)</code>, or <code>y**2-1</code>.


In [ ]:
# User-defined first-order ODE solver with MathJax verification.
try:
    import sympy as sp
except ImportError as exc:
    raise ImportError("This cell requires SymPy. Install it with %pip install sympy.") from exc

t_sym = sp.symbols("t", real=True)
a_sym, b_sym = sp.symbols("a b", positive=True, real=True)
y_fun = sp.Function("y")
y_of_t = y_fun(t_sym)

ode_examples = {
    "Linear": "t-y",
    "Logistic": "y*(1-y)",
    "Bernoulli": "y-y**2",
    "Riccati": "y**2-1",
    "Custom": "",
}
ode_example = widgets.Dropdown(options=list(ode_examples), description="Example:")
ode_rhs_input = widgets.Text(value=ode_examples["Linear"], description="f(t,y):", layout=widgets.Layout(width="75%"))
ode_t0 = widgets.FloatText(value=0.0, description="t0:")
ode_y0 = widgets.FloatText(value=1.0, description="y0:")
ode_solve_button = widgets.Button(description="Solve and verify", button_style="primary", icon="calculator")
ode_symbolic_output = widgets.Output()


def load_ode_example(change):
    expression = ode_examples[change["new"]]
    if expression:
        ode_rhs_input.value = expression


def solve_user_ode(_=None):
    """Solve the entered IVP and verify an explicit result."""
    with ode_symbolic_output:
        clear_output(wait=True)
        try:
            local_names = {
                "t": t_sym, "y": y_of_t, "a": a_sym, "b": b_sym,
                "exp": sp.exp, "sin": sp.sin, "cos": sp.cos,
                "sqrt": sp.sqrt, "Abs": sp.Abs, "log": sp.log,
            }
            rhs = sp.sympify(ode_rhs_input.value, locals=local_names)
            equation = sp.Eq(sp.diff(y_of_t, t_sym), rhs)
            t0_value = sp.Float(ode_t0.value)
            y0_value = sp.Float(ode_y0.value)
            solution = sp.dsolve(equation, ics={y_fun(t0_value): y0_value})
            display(Math(sp.latex(equation)))
            display(Math(r"\boxed{" + sp.latex(solution) + r"}"))

            if isinstance(solution, sp.Equality) and solution.lhs == y_of_t:
                candidate = solution.rhs
                residual = sp.simplify(sp.diff(candidate, t_sym) - rhs.subs(y_of_t, candidate))
                initial_residual = sp.simplify(candidate.subs(t_sym, t0_value) - y0_value)
                display(Math(r"R(t)=" + sp.latex(residual)))
                display(Math(r"y(t_0)-y_0=" + sp.latex(initial_residual)))
                if residual == 0 and initial_residual == 0:
                    display(HTML("<div style='padding:10px;background:#d1e7dd'><b>Direct symbolic verification passed.</b></div>"))
            else:
                display(HTML("<div style='padding:10px;background:#fff3cd'>An implicit solution requires implicit differentiation and a branch check.</div>"))
        except Exception as error:
            display(HTML("<div style='padding:10px;background:#f8d7da'><b>Symbolic solution failed.</b><br>" + str(error) + "</div>"))


ode_example.observe(load_ode_example, names="value")
ode_solve_button.on_click(solve_user_ode)
display(widgets.VBox([
    widgets.HBox([ode_example, ode_t0, ode_y0]),
    ode_rhs_input,
    ode_solve_button,
    ode_symbolic_output,
]))
solve_user_ode()


## 5. Euler's method and global error

Euler's recurrence is

$$
Y_{n+1}=Y_n+h\,f(t_n,Y_n).
$$

For

$$
y'=y-t^2+1,\qquad y(0)=\frac12,
$$

the exact solution is $y(t)=(t+1)^2-\tfrac12e^t$. Under standard assumptions, the global error is $O(h)$.


In [ ]:
# Compare Euler's method with the exact solution and estimate its order.
euler_steps = widgets.IntSlider(value=8, min=2, max=100, description="Steps:")
euler_horizon = widgets.FloatSlider(value=2.0, min=0.5, max=4.0, step=0.25, description="T:")
euler_button = widgets.Button(description="Run Euler method", button_style="primary", icon="play")
euler_output = widgets.Output()


def euler_rhs(t, y):
    return y - t * t + 1


def euler_exact(t):
    return (t + 1) ** 2 - 0.5 * np.exp(t)


def run_euler(_=None):
    with euler_output:
        clear_output(wait=True)
        steps, horizon = euler_steps.value, euler_horizon.value
        t_num, y_num = euler_scalar(euler_rhs, 0, 0.5, horizon, steps)
        t_fine = np.linspace(0, horizon, 1000)
        grid_error = np.max(np.abs(y_num - euler_exact(t_num)))
        counts = np.array([10, 20, 40, 80, 160])
        errors = []
        for count in counts:
            tt, yy = euler_scalar(euler_rhs, 0, 0.5, horizon, int(count))
            errors.append(np.max(np.abs(yy - euler_exact(tt))))
        order = np.polyfit(np.log(horizon / counts), np.log(errors), 1)[0]

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].plot(t_fine, euler_exact(t_fine), color="black", label="exact")
        axes[0].plot(t_num, y_num, "o-", color="#1565c0", label="Euler")
        axes[0].legend()
        axes[1].loglog(horizon / counts, errors, "o-", color="#d95f02")
        axes[1].set_xlabel("h")
        axes[1].set_ylabel("maximum error")
        plt.show()
        display(HTML(f"<b>Grid error:</b> {grid_error:.6e}<br><b>Observed order:</b> {order:.4f}"))


euler_button.on_click(run_euler)
display(widgets.VBox([widgets.HBox([euler_steps, euler_horizon]), euler_button, euler_output]))
run_euler()


## 6. Second-order characteristic roots

For $a y''+b y'+c y=0$, the characteristic equation is

$$
a r^2+b r+c=0.
$$

The sign of $D=b^2-4ac$ distinguishes two real roots, a repeated root, and a complex conjugate pair. Adjust the coefficients and initial data to see the corresponding solution.


In [ ]:
# Classify characteristic roots and plot the IVP solution.
second_a = widgets.FloatSlider(value=1.0, min=0.5, max=3.0, step=0.1, description="a:")
second_b = widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description="b:")
second_c = widgets.FloatSlider(value=1.0, min=-4.0, max=5.0, step=0.1, description="c:")
second_y0 = widgets.FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.1, description="y(0):")
second_v0 = widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description="y'(0):")
second_button = widgets.Button(description="Solve roots", button_style="info", icon="square-root-alt")
second_output = widgets.Output()


def solve_second_order(_=None):
    with second_output:
        clear_output(wait=True)
        a, b, c = second_a.value, second_b.value, second_c.value
        y0, v0 = second_y0.value, second_v0.value
        D = b * b - 4 * a * c
        t = np.linspace(0, 8, 1000)
        if D > 1e-10:
            root = np.sqrt(D)
            r1, r2 = (-b + root) / (2 * a), (-b - root) / (2 * a)
            C1 = (v0 - r2 * y0) / (r1 - r2)
            C2 = y0 - C1
            y = C1 * np.exp(r1 * t) + C2 * np.exp(r2 * t)
            result = rf"D={D:.4f}>0,\quad r_1={r1:.4f},\ r_2={r2:.4f}"
        elif D < -1e-10:
            alpha = -b / (2 * a)
            beta = np.sqrt(-D) / (2 * a)
            y = np.exp(alpha * t) * (y0 * np.cos(beta * t) + (v0 - alpha * y0) * np.sin(beta * t) / beta)
            result = rf"D={D:.4f}<0,\quad r={alpha:.4f}\pm {beta:.4f}i"
        else:
            r = -b / (2 * a)
            y = (y0 + (v0 - r * y0) * t) * np.exp(r * t)
            result = rf"D=0,\quad r={r:.4f}\text{{ repeated}}"
        fig, ax = plt.subplots()
        ax.plot(t, y, color="#1565c0", linewidth=2)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title("Characteristic-root solution")
        plt.show()
        display(Math(result))


second_button.on_click(solve_second_order)
display(widgets.VBox([
    widgets.HBox([second_a, second_b, second_c]),
    widgets.HBox([second_y0, second_v0]),
    second_button,
    second_output,
]))
solve_second_order()


## 7. Wronskian and variation of parameters

For $y''+p(t)y'+q(t)y=g(t)$, a fundamental pair satisfies

$$
W=y_1y_2'-y_1'y_2\ne0.
$$

Variation of parameters gives

$$
y_p=-y_1\int\frac{y_2g}{W}\,dt+y_2\int\frac{y_1g}{W}\,dt.
$$

For $y''+y=g(t)$, use $y_1=\cos t$, $y_2=\sin t$, and $W=1$.


In [ ]:
# Construct a variation-of-parameters solution for y''+y=g(t).
variation_forcing = widgets.Dropdown(options=["Gaussian", "Constant", "cos(2t)"], description="g(t):")
variation_horizon = widgets.FloatSlider(value=8.0, min=2.0, max=15.0, step=0.5, description="T:")
variation_button = widgets.Button(description="Build solution", button_style="primary", icon="integral")
variation_output = widgets.Output()


def variation_solution(_=None):
    with variation_output:
        clear_output(wait=True)
        t = np.linspace(0, variation_horizon.value, 5000)
        if variation_forcing.value == "Gaussian":
            forcing = np.exp(-t * t)
        elif variation_forcing.value == "Constant":
            forcing = np.ones_like(t)
        else:
            forcing = np.cos(2 * t)
        I_sin = cumulative_trapezoid(t, np.sin(t) * forcing)
        I_cos = cumulative_trapezoid(t, np.cos(t) * forcing)
        particular = -np.cos(t) * I_sin + np.sin(t) * I_cos
        dt = t[1] - t[0]
        residual = np.gradient(np.gradient(particular, dt), dt) + particular - forcing
        interior = slice(10, -10)
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].plot(t, forcing, "k--", label="g")
        axes[0].plot(t, particular, color="#1565c0", label="particular y")
        axes[0].legend()
        axes[1].plot(t[interior], residual[interior], color="#c43c39")
        axes[1].set_title("Numerical residual")
        plt.show()
        display(HTML(f"<b>Maximum interior residual:</b> {np.max(np.abs(residual[interior])):.3e}"))


variation_button.on_click(variation_solution)
display(widgets.VBox([widgets.HBox([variation_forcing, variation_horizon]), variation_button, variation_output]))
variation_solution()


## 8. Power-series solutions

At an ordinary point, analytic coefficients give an analytic solution. For $y''+y=0$,

$$
a_{n+2}=-\frac{a_n}{(n+2)(n+1)}.
$$

For $y''-xy=0$,

$$
a_2=0,\qquad
a_{n+2}=\frac{a_{n-1}}{(n+2)(n+1)}.
$$

Generate coefficients and view the truncated polynomial with MathJax.


In [ ]:
# Generate power-series coefficients and render the polynomial.
series_equation = widgets.Dropdown(options=["y''+y=0", "y''-xy=0"], description="Equation:")
series_a0 = widgets.FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.1, description="a0:")
series_a1 = widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1, description="a1:")
series_order = widgets.IntSlider(value=12, min=2, max=30, description="Order:")
series_button = widgets.Button(description="Generate series", button_style="info", icon="list-ol")
series_output = widgets.Output()


def polynomial_latex(coefficients):
    terms = []
    for n, value in enumerate(coefficients):
        if abs(value) < 1e-13:
            continue
        core = f"{abs(value):.6g}"
        if n == 1:
            core += "x"
        elif n > 1:
            core += rf"x^{{{n}}}"
        terms.append((("-" if value < 0 else "") if not terms else (" - " if value < 0 else " + ")) + core)
    return "".join(terms) if terms else "0"


def generate_series(_=None):
    with series_output:
        clear_output(wait=True)
        order = series_order.value
        coefficients = np.zeros(order + 1)
        coefficients[0], coefficients[1] = series_a0.value, series_a1.value
        if series_equation.value == "y''+y=0":
            for n in range(order - 1):
                coefficients[n + 2] = -coefficients[n] / ((n + 2) * (n + 1))
        else:
            coefficients[2] = 0
            for n in range(1, order - 1):
                coefficients[n + 2] = coefficients[n - 1] / ((n + 2) * (n + 1))
        x = np.linspace(-3, 3, 800)
        approximation = np.polynomial.polynomial.polyval(x, coefficients)
        fig, ax = plt.subplots()
        ax.plot(x, approximation, color="#1565c0", label="series")
        if series_equation.value == "y''+y=0":
            ax.plot(x, series_a0.value * np.cos(x) + series_a1.value * np.sin(x), "k--", label="exact")
        ax.legend()
        plt.show()
        display(Math(r"y_N(x)=" + polynomial_latex(coefficients)))


series_button.on_click(generate_series)
display(widgets.VBox([
    widgets.HBox([series_equation, series_order]),
    widgets.HBox([series_a0, series_a1]),
    series_button,
    series_output,
]))
generate_series()


## 9. Linear systems and matrix exponentials

The solution of

$$
\mathbf y'=A\mathbf y,\qquad \mathbf y(0)=\mathbf y_0,
$$

is

$$
\mathbf y(t)=e^{tA}\mathbf y_0.
$$

The eigenvalues and Jordan structure determine the phase portrait.


In [ ]:
# Explore four representative matrix-exponential flows.
system_matrix = widgets.Dropdown(options=["Rotation", "Saddle", "Stable spiral", "Jordan block"], description="Matrix:")
system_x0 = widgets.FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.1, description="x0:")
system_y0 = widgets.FloatSlider(value=0.3, min=-2.0, max=2.0, step=0.1, description="y0:")
system_button = widgets.Button(description="Plot matrix flow", button_style="primary", icon="vector-square")
system_output = widgets.Output()


def matrix_flow(name, t, x0, y0):
    if name == "Rotation":
        return x0*np.cos(t)+y0*np.sin(t), -x0*np.sin(t)+y0*np.cos(t), r"A=\begin{pmatrix}0&1\\-1&0\end{pmatrix}"
    if name == "Saddle":
        return x0*np.exp(t), y0*np.exp(-t), r"A=\begin{pmatrix}1&0\\0&-1\end{pmatrix}"
    if name == "Stable spiral":
        d = np.exp(-0.3*t)
        return d*(x0*np.cos(t)+y0*np.sin(t)), d*(-x0*np.sin(t)+y0*np.cos(t)), r"A=\begin{pmatrix}-0.3&1\\-1&-0.3\end{pmatrix}"
    g = np.exp(0.25*t)
    return g*(x0+t*y0), g*y0, r"A=\begin{pmatrix}0.25&1\\0&0.25\end{pmatrix}"


def plot_system(_=None):
    with system_output:
        clear_output(wait=True)
        t = np.linspace(0, 10, 1500)
        x, y, matrix = matrix_flow(system_matrix.value, t, system_x0.value, system_y0.value)
        fig, ax = plt.subplots()
        ax.plot(x, y, color="#1565c0")
        ax.scatter([x[0]], [y[0]], color="#c43c39")
        ax.axis("equal")
        ax.set_title("Matrix-exponential trajectory")
        plt.show()
        display(Math(matrix))


system_button.on_click(plot_system)
display(widgets.VBox([widgets.HBox([system_matrix, system_x0, system_y0]), system_button, system_output]))
plot_system()


## 10. Characteristics and PDE classification

For

$$
u_t+c\,u_x=-\lambda u,\qquad u(x,0)=f(x),
$$

characteristics give $u(x,t)=e^{-\lambda t}f(x-ct)$. For

$$
A u_{xx}+2B u_{xy}+C u_{yy},
$$

the sign of $\Delta=B^2-AC$ gives the hyperbolic, parabolic, or elliptic classification.


In [ ]:
# Switch between transport characteristics and PDE classification.
pde_mode = widgets.ToggleButtons(options=["Transport", "Classification"], description="Mode:")
transport_c = widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1, description="c:")
transport_lambda = widgets.FloatSlider(value=0.3, min=-1.0, max=2.0, step=0.1, description="lambda:")
transport_time = widgets.FloatSlider(value=1.5, min=0.0, max=5.0, step=0.1, description="time:")
pde_A = widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1, description="A:")
pde_B = widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description="B:")
pde_C = widgets.FloatSlider(value=-1.0, min=-3.0, max=3.0, step=0.1, description="C:")
pde_button = widgets.Button(description="Analyze PDE", button_style="info", icon="search")
pde_output = widgets.Output()


def analyze_pde(_=None):
    with pde_output:
        clear_output(wait=True)
        if pde_mode.value == "Transport":
            x = np.linspace(-8, 8, 1200)
            initial = np.exp(-x*x)
            solution = np.exp(-transport_lambda.value*transport_time.value) * np.exp(-(x-transport_c.value*transport_time.value)**2)
            fig, ax = plt.subplots()
            ax.plot(x, initial, "k--", label="initial")
            ax.plot(x, solution, color="#1565c0", label="u(x,t)")
            ax.legend()
            plt.show()
            display(Math(r"u(x,t)=e^{-\lambda t}f(x-ct)"))
        else:
            delta = pde_B.value**2 - pde_A.value*pde_C.value
            if delta > 1e-10:
                name, color = "hyperbolic", "#d1e7dd"
            elif delta < -1e-10:
                name, color = "elliptic", "#cfe2ff"
            else:
                name, color = "parabolic", "#fff3cd"
            display(Math(rf"\Delta={delta:.6f}"))
            display(HTML(f"<div style='padding:14px;background:{color}'><b>{name}</b></div>"))


pde_button.on_click(analyze_pde)
display(widgets.VBox([
    pde_mode,
    widgets.HBox([transport_c, transport_lambda, transport_time]),
    widgets.HBox([pde_A, pde_B, pde_C]),
    pde_button,
    pde_output,
]))
analyze_pde()


## 11. Heat equation on a finite interval

For zero boundary data on $(0,\pi)$,

$$
u(x,t)=\sum_{n=1}^{\infty}b_ne^{-\kappa n^2t}\sin(nx).
$$

Higher modes decay faster, and the energy $\int_0^\pi u^2\,dx$ decreases.


In [ ]:
# Animate two decaying Fourier sine modes.
finite_heat_time = widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.02, description="time:")
finite_heat_kappa = widgets.FloatSlider(value=0.3, min=0.05, max=1.0, step=0.05, description="kappa:")
finite_heat_b1 = widgets.FloatSlider(value=3.0, min=-4.0, max=4.0, step=0.25, description="b1:")
finite_heat_b3 = widgets.FloatSlider(value=-2.0, min=-4.0, max=4.0, step=0.25, description="b3:")
finite_heat_button = widgets.Button(description="Evolve modes", button_style="primary", icon="fire")
finite_heat_output = widgets.Output()


def finite_heat(_=None):
    with finite_heat_output:
        clear_output(wait=True)
        x = np.linspace(0, PI, 1000)
        t, k = finite_heat_time.value, finite_heat_kappa.value
        initial = finite_heat_b1.value*np.sin(x) + finite_heat_b3.value*np.sin(3*x)
        solution = finite_heat_b1.value*np.exp(-k*t)*np.sin(x) + finite_heat_b3.value*np.exp(-9*k*t)*np.sin(3*x)
        energy = np.trapezoid(solution**2, x)
        fig, ax = plt.subplots()
        ax.plot(x, initial, "k--", label="initial")
        ax.plot(x, solution, color="#d7301f", label="u(x,t)")
        ax.legend()
        plt.show()
        display(HTML(f"<b>Current energy:</b> {energy:.7f}"))


finite_heat_button.on_click(finite_heat)
display(widgets.VBox([
    widgets.HBox([finite_heat_time, finite_heat_kappa, finite_heat_b1, finite_heat_b3]),
    finite_heat_button,
    finite_heat_output,
]))
finite_heat()


## 12. Heat equation on the real line

The heat-kernel formula is

$$
u(x,t)=\frac1{\sqrt{4\pi\kappa t}}
\int_{\mathbb R}e^{-(x-y)^2/(4\kappa t)}f(y)\,dy,
$$

and in frequency space $\widehat u=e^{-\kappa t\omega^2}\widehat f$. This multiplier smooths nonsmooth data and preserves mass.


In [ ]:
# Apply the real-line heat multiplier to selected initial data.
real_heat_initial = widgets.Dropdown(options=["Box", "Two pulses", "Noisy Gaussian"], description="Initial:")
real_heat_time = widgets.FloatSlider(value=0.15, min=0.01, max=1.5, step=0.01, description="time:")
real_heat_kappa = widgets.FloatSlider(value=0.5, min=0.1, max=1.5, step=0.1, description="kappa:")
real_heat_button = widgets.Button(description="Apply heat kernel", button_style="primary", icon="fire")
real_heat_output = widgets.Output()


def real_heat(_=None):
    with real_heat_output:
        clear_output(wait=True)
        x = time_grid(18, 2**14)
        if real_heat_initial.value == "Box":
            initial = (np.abs(x) <= 2).astype(float)
        elif real_heat_initial.value == "Two pulses":
            initial = np.exp(-3*(x-3)**2) + 0.8*np.exp(-5*(x+3)**2)
        else:
            initial = np.exp(-0.4*x*x)*(1+0.3*np.cos(10*x))
        omega, initial_hat = continuous_fft(x, initial)
        multiplier = np.exp(-real_heat_kappa.value*real_heat_time.value*omega**2)
        solution = np.real(continuous_ifft(omega, initial_hat*multiplier))
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].plot(x, initial, "k--", label="initial")
        axes[0].plot(x, solution, color="#d7301f", label="solution")
        axes[0].set_xlim(-8, 8)
        axes[0].legend()
        axes[1].plot(omega, np.abs(initial_hat), color="#636363")
        axes[1].plot(omega, np.abs(initial_hat*multiplier), color="#2ca25f")
        axes[1].set_xlim(-20, 20)
        plt.show()
        display(HTML(f"<b>Mass difference:</b> {abs(np.trapezoid(initial,x)-np.trapezoid(solution,x)):.3e}"))


real_heat_button.on_click(real_heat)
display(widgets.VBox([widgets.HBox([real_heat_initial, real_heat_time, real_heat_kappa]), real_heat_button, real_heat_output]))
real_heat()


## 13. Wave equation and finite propagation

d'Alembert's formula is

$$
u(x,t)=\frac12[f(x-ct)+f(x+ct)]
+\frac1{2c}\int_{x-ct}^{x+ct}g(s)\,ds.
$$

It shows finite propagation speed. On a fixed string, Fourier sine modes oscillate rather than decay.


In [ ]:
# Compare a travelling pulse with a standing string mode.
wave_mode = widgets.ToggleButtons(options=["Real-line pulse", "Finite string"], description="Problem:")
wave_speed = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description="c:")
wave_time = widgets.FloatSlider(value=1.5, min=0.0, max=5.0, step=0.1, description="time:")
wave_number = widgets.IntSlider(value=2, min=1, max=8, description="mode:")
wave_button = widgets.Button(description="Propagate wave", button_style="info", icon="water")
wave_output = widgets.Output()


def wave_solution(_=None):
    with wave_output:
        clear_output(wait=True)
        c, t = wave_speed.value, wave_time.value
        if wave_mode.value == "Real-line pulse":
            x = np.linspace(-10, 10, 1600)
            initial = np.exp(-x*x)
            solution = 0.5*np.exp(-(x-c*t)**2) + 0.5*np.exp(-(x+c*t)**2)
        else:
            x = np.linspace(0, PI, 1200)
            n = wave_number.value
            initial = np.sin(n*x)
            solution = np.cos(c*n*t)*np.sin(n*x)
        fig, ax = plt.subplots()
        ax.plot(x, initial, "k--", label="initial")
        ax.plot(x, solution, color="#1565c0", label="u(x,t)")
        ax.legend()
        plt.show()


wave_button.on_click(wave_solution)
display(widgets.VBox([wave_mode, widgets.HBox([wave_speed, wave_time, wave_number]), wave_button, wave_output]))
wave_solution()


## 14. Laplace's equation on a rectangle

With top boundary $\sin(n\pi x/a)$ and zero values on the other sides,

$$
u(x,y)=
\frac{\sinh(n\pi y/a)}{\sinh(n\pi b/a)}
\sin\left(\frac{n\pi x}{a}\right)
$$

solves $u_{xx}+u_{yy}=0$. The maximum principle gives uniqueness.


In [ ]:
# Visualize a separated harmonic mode and check its Laplacian.
laplace_mode = widgets.IntSlider(value=2, min=1, max=7, description="n:")
laplace_width = widgets.FloatSlider(value=2.0, min=1.0, max=4.0, step=0.1, description="a:")
laplace_height = widgets.FloatSlider(value=1.5, min=0.5, max=3.0, step=0.1, description="b:")
laplace_button = widgets.Button(description="Draw harmonic mode", button_style="primary", icon="border-all")
laplace_output = widgets.Output()


def laplace_rectangle(_=None):
    with laplace_output:
        clear_output(wait=True)
        n, a, b = laplace_mode.value, laplace_width.value, laplace_height.value
        x = np.linspace(0, a, 240)
        y = np.linspace(0, b, 180)
        X, Y = np.meshgrid(x, y)
        u = np.sinh(n*PI*Y/a)/np.sinh(n*PI*b/a)*np.sin(n*PI*X/a)
        dx, dy = x[1]-x[0], y[1]-y[0]
        residual = np.gradient(np.gradient(u,dx,axis=1),dx,axis=1) + np.gradient(np.gradient(u,dy,axis=0),dy,axis=0)
        fig, ax = plt.subplots()
        contour = ax.contourf(X, Y, u, levels=30, cmap="coolwarm")
        fig.colorbar(contour, ax=ax)
        plt.show()
        display(HTML(f"<b>Maximum interior Laplacian:</b> {np.max(np.abs(residual[3:-3,3:-3])):.3e}"))


laplace_button.on_click(laplace_rectangle)
display(widgets.VBox([widgets.HBox([laplace_mode, laplace_width, laplace_height]), laplace_button, laplace_output]))
laplace_rectangle()


## 15. Black–Scholes as a heat equation

The European call formula obtained from the heat-equation reduction is

$$
V=S\,N(d_1)-Ke^{-r\tau}N(d_2),
$$

where

$$
d_1=\frac{\log(S/K)+(r+\sigma^2/2)\tau}{\sigma\sqrt{\tau}},
\qquad
d_2=d_1-\sigma\sqrt{\tau}.
$$


In [ ]:
# Interactive Black-Scholes European call explorer.
bs_stock = widgets.FloatSlider(value=100, min=20, max=200, step=1, description="S:")
bs_strike = widgets.FloatSlider(value=100, min=40, max=160, step=1, description="K:")
bs_rate = widgets.FloatSlider(value=0.03, min=-0.02, max=0.15, step=0.005, description="r:")
bs_sigma = widgets.FloatSlider(value=0.20, min=0.05, max=0.80, step=0.01, description="sigma:")
bs_tau = widgets.FloatSlider(value=1.0, min=0.02, max=3.0, step=0.02, description="tau:")
bs_button = widgets.Button(description="Price call", button_style="success", icon="chart-line")
bs_output = widgets.Output()


def call_values(S, K, r, sigma, tau):
    d1 = (np.log(S/K)+(r+0.5*sigma**2)*tau)/(sigma*np.sqrt(tau))
    d2 = d1-sigma*np.sqrt(tau)
    price = S*normal_cdf(d1)-K*np.exp(-r*tau)*normal_cdf(d2)
    delta = normal_cdf(d1)
    gamma = np.exp(-0.5*d1**2)/np.sqrt(TWO_PI)/(S*sigma*np.sqrt(tau))
    return price, delta, gamma, d1, d2


def price_call(_=None):
    with bs_output:
        clear_output(wait=True)
        K, r, sigma, tau = bs_strike.value, bs_rate.value, bs_sigma.value, bs_tau.value
        S_grid = np.linspace(1, max(220,2.2*K), 600)
        prices = call_values(S_grid,K,r,sigma,tau)[0]
        price, delta, gamma, d1, d2 = call_values(bs_stock.value,K,r,sigma,tau)
        fig, ax = plt.subplots()
        ax.plot(S_grid, prices, color="#1565c0", label="call")
        ax.plot(S_grid, np.maximum(S_grid-K,0), "k--", label="payoff")
        ax.scatter([bs_stock.value],[price],color="#c43c39")
        ax.legend()
        plt.show()
        display(Math(rf"d_1={float(d1):.6f},\quad d_2={float(d2):.6f}"))
        display(Math(rf"V={float(price):.6f},\quad \Delta={float(delta):.6f},\quad \Gamma={float(gamma):.6g}"))


bs_button.on_click(price_call)
display(widgets.VBox([
    widgets.HBox([bs_stock,bs_strike,bs_tau]),
    widgets.HBox([bs_rate,bs_sigma]),
    bs_button,
    bs_output,
]))
price_call()


## 16. Explicit finite differences and the CFL restriction

The explicit heat update is

$$
U_i^{n+1}
=\lambda U_{i-1}^n+(1-2\lambda)U_i^n+\lambda U_{i+1}^n,
\qquad
\lambda=\frac{\kappa\Delta t}{(\Delta x)^2}.
$$

Stability requires $0\le\lambda\le\tfrac12$. Above this threshold, high-frequency errors can grow.


In [ ]:
# Demonstrate stable and unstable explicit heat updates.
cfl_lambda = widgets.FloatSlider(value=0.45, min=0.05, max=0.80, step=0.01, description="lambda:")
cfl_points = widgets.IntSlider(value=50, min=20, max=100, step=5, description="N:")
cfl_steps = widgets.IntSlider(value=120, min=10, max=300, step=10, description="Steps:")
cfl_button = widgets.Button(description="Run scheme", button_style="warning", icon="calculator")
cfl_output = widgets.Output()


def run_heat_scheme(_=None):
    with cfl_output:
        clear_output(wait=True)
        lam, N, steps = cfl_lambda.value, cfl_points.value, cfl_steps.value
        x = np.linspace(0,1,N+2)
        current = np.sin(PI*x) + 0.02*np.sin(N*PI*x/(N+1))
        snapshots = [(0,current.copy())]
        for step in range(1,steps+1):
            new = current.copy()
            new[1:-1] = lam*current[:-2] + (1-2*lam)*current[1:-1] + lam*current[2:]
            new[0] = new[-1] = 0
            current = new
            if step in {steps//4,steps//2,steps}:
                snapshots.append((step,current.copy()))
        fig, ax = plt.subplots()
        for step, values in snapshots:
            ax.plot(x,values,label=f"step {step}")
        ax.legend()
        plt.show()
        amplification = 1-4*lam*np.sin(N*PI/(2*(N+1)))**2
        stable = lam <= 0.5
        color = "#d1e7dd" if stable else "#f8d7da"
        display(HTML(f"<div style='padding:12px;background:{color}'><b>{'Stable' if stable else 'Unstable'} CFL range</b><br>Highest-mode factor: {amplification:.6f}<br>Final maximum: {np.max(np.abs(current)):.6g}</div>"))


cfl_button.on_click(run_heat_scheme)
display(widgets.VBox([widgets.HBox([cfl_lambda,cfl_points,cfl_steps]),cfl_button,cfl_output]))
run_heat_scheme()


## 17. Formal discovery and direct verification

Dividing by the unknown can lose equilibria, and transform methods may be used formally on non-decaying data. The decisive step is substitution into the original equation.

$$
\begin{aligned}
y'&=y(1-y), &&y\equiv0,\\
u_t&=\kappa u_{xx}, &&u=e^{ax+a^2\kappa t},\\
u_{xx}+u_{yy}&=0, &&u\equiv1.
\end{aligned}
$$


In [ ]:
# Verify candidates that formal operations may discover or lose.
verify_example = widgets.Dropdown(options=["Lost equilibrium","Non-integrable heat solution","Constant harmonic solution"],description="Example:")
verify_a = widgets.FloatSlider(value=0.6,min=-1.5,max=1.5,step=0.1,description="a:")
verify_kappa = widgets.FloatSlider(value=0.8,min=0.1,max=2.0,step=0.1,description="kappa:")
verify_time = widgets.FloatSlider(value=0.7,min=0.0,max=2.0,step=0.1,description="time:")
verify_button = widgets.Button(description="Verify directly",button_style="primary",icon="check-double")
verify_output = widgets.Output()


def direct_verification(_=None):
    with verify_output:
        clear_output(wait=True)
        x = np.linspace(-4,4,1000)
        if verify_example.value == "Lost equilibrium":
            candidate = np.zeros_like(x)
            residual = candidate-candidate*(1-candidate)
            formula = r"y\equiv0,\quad y'-y(1-y)=0"
        elif verify_example.value == "Non-integrable heat solution":
            a,k,t = verify_a.value,verify_kappa.value,verify_time.value
            candidate = np.exp(a*x+a*a*k*t)
            residual = a*a*k*candidate-k*a*a*candidate
            formula = r"u=e^{ax+a^2\kappa t},\quad u_t-\kappa u_{xx}=0"
        else:
            candidate = np.ones_like(x)
            residual = np.zeros_like(x)
            formula = r"u\equiv1,\quad u_{xx}+u_{yy}=0"
        fig, ax = plt.subplots()
        ax.plot(x,candidate,color="#1565c0")
        plt.show()
        display(Math(formula))
        display(HTML(f"<b>Maximum residual:</b> {np.max(np.abs(residual)):.3e}"))


verify_button.on_click(direct_verification)
display(widgets.VBox([widgets.HBox([verify_example,verify_a,verify_kappa,verify_time]),verify_button,verify_output]))
direct_verification()


## 18. Theorem and method selector

Choose the desired conclusion before choosing a theorem:

$$
\text{existence},\quad
\text{uniqueness},\quad
\text{representation},\quad
\text{stability},\quad
\text{verification}.
$$


In [ ]:
# Recommend a theorem or validation principle.
method_data = {
    "Local existence": ("Peano theorem","Continuity gives at least one local solution, but not uniqueness."),
    "Local existence and uniqueness": ("Picard–Lindelof theorem","Continuity plus a local Lipschitz condition in y gives a unique local solution."),
    "Second-order fundamental basis": ("Wronskian and Abel identity","A nonzero Wronskian at one point remains nonzero for solutions of the same linear equation."),
    "Constant linear system": ("Matrix exponential","Use y(t)=exp((t-t0)A)y0 and verify by differentiation."),
    "PDE classification": ("Principal discriminant","For A u_xx+2B u_xy+C u_yy, inspect B^2-AC."),
    "Explicit heat stability": ("CFL criterion","Require 0<=lambda<=1/2 in one space dimension."),
    "Formal transform or substitution": ("Direct verification","Check regularity, the original equation, all data, and a separate uniqueness theorem."),
}
method_goal = widgets.Dropdown(options=list(method_data),description="Goal:",layout=widgets.Layout(width="85%"))
method_button = widgets.Button(description="Recommend",button_style="primary",icon="book-open")
method_output = widgets.Output()


def recommend_method(_=None):
    with method_output:
        clear_output(wait=True)
        title,text = method_data[method_goal.value]
        display(HTML(f"<div style='padding:12px;border:1px solid #aaccee'><h4>{title}</h4>{text}</div>"))


method_button.on_click(recommend_method)
display(widgets.VBox([method_goal,method_button,method_output]))
recommend_method()


## 19. Concept quiz

The quiz checks existence versus uniqueness, lost solutions, ODE and PDE classifications, and numerical stability.


In [ ]:
# Self-checking conceptual quiz.
quiz_questions = [
    ("Continuity of f alone guarantees:",["local existence only","uniqueness","global existence"],0,"Peano gives existence, while uniqueness needs more."),
    ("Picard iteration uses:",["the integral equation","Fourier inversion","a characteristic polynomial"],0,"It is a fixed-point iteration for the Volterra equation."),
    ("Division by B(y) may:",["lose equilibria","guarantee uniqueness","create a PDE"],0,"Zeros of B must be checked before division."),
    ("Euler's global order is:",["one","two","four"],0,"Its standard global error is O(h)."),
    ("Hyperbolicity means:",["B^2-AC>0","B^2-AC=0","B^2-AC<0"],0,"A positive discriminant gives hyperbolicity."),
    ("Heat flow typically:",["smooths high frequencies","has finite propagation","amplifies all modes"],0,"The Gaussian multiplier damps high frequencies."),
    ("Wave propagation is:",["finite speed","instantaneous everywhere","not an IVP"],0,"d'Alembert's formula gives a characteristic cone."),
    ("Explicit heat stability needs:",["lambda<=1/2","lambda>1","no restriction"],0,"The one-dimensional CFL bound is one half."),
    ("A formal candidate is proved by:",["direct verification","its appearance","ignoring its domain"],0,"Substitution and data checks complete the proof."),
]
quiz_boxes = []
for number,(question,options,_,_) in enumerate(quiz_questions,start=1):
    radio = widgets.RadioButtons(options=options,description=f"{number}.",layout=widgets.Layout(width="95%"))
    quiz_boxes.append(widgets.VBox([widgets.HTML(f"<b>{question}</b>"),radio]))
quiz_button = widgets.Button(description="Check answers",button_style="success",icon="check-circle")
quiz_output = widgets.Output()


def grade_quiz(_=None):
    with quiz_output:
        clear_output(wait=True)
        score = 0
        feedback = []
        for number,((question,options,correct,explanation),box) in enumerate(zip(quiz_questions,quiz_boxes),start=1):
            if box.children[1].value == options[correct]:
                score += 1
                feedback.append(f"<span style='color:#198754'>✓ {number}. Correct.</span>")
            else:
                feedback.append(f"<span style='color:#b02a37'>✗ {number}. {explanation}</span>")
        display(HTML(f"<h4>Score: {score}/{len(quiz_questions)}</h4>"+"<br>".join(feedback)))


quiz_button.on_click(grade_quiz)
display(widgets.VBox(quiz_boxes+[quiz_button,quiz_output]))


## 20. Practice generator

For every problem: identify the method, derive the candidate, determine its domain, verify the equation and data, and justify uniqueness or stability separately.


In [ ]:
# Generate practice problems with solution guides.
practice_bank = {
    "First-order ODEs":[
        ("Solve y'+2y=1, y(0)=0.","Use exp(2t). The solution is (1-exp(-2t))/2 on the real line."),
        ("Find all solutions of y'=y(1-y).","Record y=0 and y=1 before division; then integrate dy/[y(1-y)]=dt."),
    ],
    "Second-order and series":[
        ("Solve y''-2y'+5y=0.","Roots are 1±2i, so y=e^t(C1 cos2t+C2 sin2t)."),
        ("Find the recurrence for y''-xy=0.","a2=0 and a_(n+2)=a_(n-1)/[(n+2)(n+1)]."),
    ],
    "PDEs":[
        ("Solve u_t+c u_x=0 with u(x,0)=f(x).","Characteristics give u(x,t)=f(x-ct)."),
        ("Classify u_xx-4u_xy+3u_yy=0.","A=1, B=-2, C=3, so B^2-AC=1>0: hyperbolic."),
    ],
    "Numerical methods":[
        ("Why is lambda=0.6 unsafe for explicit heat flow?","The highest-mode factor approaches 1-4lambda=-1.4, whose magnitude exceeds one."),
        ("Why is Euler exact for y'=constant?","The exact solution is affine, so each tangent increment is exact."),
    ],
    "Modelling":[
        ("Check the terminal limit of the Black–Scholes call.","As tau tends to zero, the normal terms select S>K and S<K, giving (S-K)^+."),
        ("Give a heat solution outside classical transform hypotheses.","u=exp(ax+a^2 kappa t) solves the PDE directly, although exp(ax) is not in L1 or L2."),
    ],
}
practice_topic = widgets.Dropdown(options=list(practice_bank),description="Topic:")
practice_new = widgets.Button(description="New problem",button_style="primary",icon="random")
practice_reveal = widgets.Button(description="Reveal guide",button_style="warning",icon="lightbulb")
practice_problem = widgets.HTMLMath()
practice_solution = widgets.HTMLMath()
practice_state = {"solution":""}
practice_rng = np.random.default_rng(20260717)


def new_problem(_=None):
    choices = practice_bank[practice_topic.value]
    problem,solution = choices[int(practice_rng.integers(0,len(choices)))]
    practice_state["solution"] = solution
    practice_problem.value = f"<div style='padding:12px;background:#eef6ff'><b>Problem.</b> {problem}</div>"
    practice_solution.value = ""


def reveal_solution(_=None):
    practice_solution.value = f"<div style='padding:12px;background:#fff4d6'><b>Guide.</b> {practice_state['solution']}</div>"


practice_new.on_click(new_problem)
practice_reveal.on_click(reveal_solution)
practice_topic.observe(new_problem,names="value")
display(widgets.VBox([widgets.HBox([practice_topic,practice_new,practice_reveal]),practice_problem,practice_solution]))
new_problem()


## Suggested learning path and final checkpoint

1. Connect slope fields, integral equations, Picard iteration, existence, and uniqueness.
2. Use the symbolic input cell, but inspect branches and maximal intervals.
3. Compare exact methods with Euler approximation.
4. Connect roots, Wronskians, power series, and matrix exponentials.
5. Contrast transport, heat, wave, and Laplace equations.
6. Cross the CFL threshold and observe instability.
7. Finish every formal method with direct verification.

$$
\boxed{
\text{existence}
+\text{uniqueness}
+\text{domain}
+\text{computation}
+\text{verification}
}
$$
